<a href="https://www.kaggle.com/code/nmavros/deepfire-forecaster?scriptVersionId=300756183" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import os
import glob
import torch
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, random_split

print("Initializing environment and defining the Dataset class...")

# Set random seed for reproducibility in research
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    
set_seed(42)

class SatFireDataset(Dataset):
    def __init__(self, event_dirs, target_max=100.0):
        self.event_dirs = event_dirs
        self.target_max = target_max

    def __len__(self):
        return len(self.event_dirs)

    def __getitem__(self, idx):
        event_path = self.event_dirs[idx]
        
        # Load VIIRS Thermal Sequence (Days 1 to 3)
        viirs_files = sorted(glob.glob(os.path.join(event_path, "VIIRS", "*.tif")))
        viirs_seq = []
        for f in viirs_files[:3]: 
            with rasterio.open(f) as src:
                viirs_seq.append(src.read())
        viirs_seq = np.array(viirs_seq) 
        
        # Load LULC (Land Use / Land Cover)
        lulc_file = glob.glob(os.path.join(event_path, "LULC", "*.tif"))[0]
        with rasterio.open(lulc_file) as src:
            lulc = src.read()
            
        # Load Target (FirePred Day 4)
        target_file = glob.glob(os.path.join(event_path, "FirePred", "*.tif"))[0]
        with rasterio.open(target_file) as src:
            target = src.read(4) 
            
        # Data Normalization & Signal Clipping
        target = np.nan_to_num(target, nan=0.0)
        viirs_seq = np.nan_to_num(viirs_seq, nan=0.0)
        lulc = np.nan_to_num(lulc, nan=0.0)
        
        # Clip outliers and apply Min-Max Scaling [0, 1]
        target = np.clip(target, a_min=0.0, a_max=self.target_max)
        target = target / self.target_max
        
        viirs_tensor = torch.tensor(viirs_seq, dtype=torch.float32)
        lulc_tensor = torch.tensor(lulc, dtype=torch.float32)
        target_tensor = torch.tensor(target, dtype=torch.float32).unsqueeze(0) 
        
        return viirs_tensor, lulc_tensor, target_tensor

print("Environment setup complete. Dataset class is ready.")

Initializing environment and defining the Dataset class...
Environment setup complete. Dataset class is ready.


In [2]:
import os
import torch
from torch.utils.data import DataLoader, random_split

print("Scanning dataset directories and creating DataLoaders...\n")

KAGGLE_DATA_ROOT = "/kaggle/input/ts-satfire/ts-satfire"

# 1. Safely collect all valid event directories
all_event_dirs = sorted([
    os.path.join(KAGGLE_DATA_ROOT, d) for d in os.listdir(KAGGLE_DATA_ROOT) 
    if os.path.isdir(os.path.join(KAGGLE_DATA_ROOT, d))
])

total_events = len(all_event_dirs)
print(f"Total fire events found: {total_events}")

# 2. Define the split ratios (70% Train, 15% Validation, 15% Test)
train_size = int(0.7 * total_events)
val_size = int(0.15 * total_events)
test_size = total_events - train_size - val_size

# 3. Split the dataset randomly but consistently (using the fixed seed)
generator = torch.Generator().manual_seed(42)
train_dirs, val_dirs, test_dirs = random_split(
    all_event_dirs, 
    [train_size, val_size, test_size], 
    generator=generator
)

# 4. Instantiate the Normalized Datasets
# Assuming 100.0 is a safe maximum for normalization based on our previous profiling
TARGET_MAX = 100.0
train_dataset = SatFireDataset(train_dirs, target_max=TARGET_MAX)
val_dataset = SatFireDataset(val_dirs, target_max=TARGET_MAX)
test_dataset = SatFireDataset(test_dirs, target_max=TARGET_MAX)

# 5. Create the DataLoaders
BATCH_SIZE = 8
# drop_last=True prevents crashes if the last batch is smaller than 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

print("-" * 50)
print("DATASET SPLIT SUMMARY")
print("-" * 50)
print(f"Training samples   : {len(train_dataset)} (Batches: {len(train_loader)})")
print(f"Validation samples : {len(val_dataset)} (Batches: {len(val_loader)})")
print(f"Testing samples    : {len(test_dataset)} (Batches: {len(test_loader)})")
print("-" * 50)
print("DataLoaders are successfully initialized and ready.")

Scanning dataset directories and creating DataLoaders...

Total fire events found: 192
--------------------------------------------------
DATASET SPLIT SUMMARY
--------------------------------------------------
Training samples   : 134 (Batches: 16)
Validation samples : 28 (Batches: 4)
Testing samples    : 30 (Batches: 30)
--------------------------------------------------
DataLoaders are successfully initialized and ready.


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print("Initializing the Hybrid CNN-Transformer Architecture...")

class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class SpatialEncoder(nn.Module):
    def __init__(self, viirs_channels=8, lulc_channels=1, embed_dim=128):
        super().__init__()
        in_channels = viirs_channels + lulc_channels
        self.layer1 = CNNBlock(in_channels, 32)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.layer2 = CNNBlock(32, 64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.layer3 = CNNBlock(64, embed_dim)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, viirs_seq, lulc):
        B, T, C_v, H, W = viirs_seq.shape
        C_l = lulc.shape[1]
        
        viirs_reshaped = viirs_seq.view(B * T, C_v, H, W)
        lulc_repeated = lulc.unsqueeze(1).repeat(1, T, 1, 1, 1)
        lulc_reshaped = lulc_repeated.view(B * T, C_l, H, W)
        
        x = torch.cat([viirs_reshaped, lulc_reshaped], dim=1)
        
        x = self.pool1(self.layer1(x))
        x = self.pool2(self.layer2(x))
        features = self.pool3(self.layer3(x))
        
        _, C_f, H_f, W_f = features.shape
        return features.view(B, T, C_f, H_f, W_f)

class SimpleSpatiotemporalTransformer(nn.Module):
    def __init__(self, embed_dim=128, num_heads=4, num_layers=2, seq_len=3, spatial_size=32):
        super().__init__()
        self.num_tokens = seq_len * spatial_size * spatial_size
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_tokens, embed_dim) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim * 2,
            dropout=0.1, activation='gelu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, features_temporal):
        B, T, C, H, W = features_temporal.shape
        x = features_temporal.view(B, T * H * W, C)
        x = x + self.pos_embed
        x = self.transformer(x)
        return x.view(B, T, C, H, W)

class TemporalAttentionPooling(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.attention_net = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 2, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // 2, 1, kernel_size=1)
        )

    def forward(self, x):
        B, T, C, H, W = x.shape
        x_reshaped = x.view(B * T, C, H, W)
        scores = self.attention_net(x_reshaped).view(B, T, -1).mean(dim=2)
        weights = F.softmax(scores, dim=1)
        weights_expanded = weights.view(B, T, 1, 1, 1)
        pooled_features = torch.sum(x * weights_expanded, dim=1)
        return pooled_features, weights

class GenerativeDecoder(nn.Module):
    def __init__(self, in_channels=128):
        super().__init__()
        self.up1 = nn.Sequential(nn.ConvTranspose2d(in_channels, 64, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.up2 = nn.Sequential(nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.up3 = nn.Sequential(nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True))
        self.final_conv = nn.Conv2d(16, 1, kernel_size=3, padding=1)
        
    def forward(self, x):
        return self.final_conv(self.up3(self.up2(self.up1(x))))

class SatGenTransformerNormalized(nn.Module):
    def __init__(self, embed_dim=128):
        super().__init__()
        self.spatial_encoder = SpatialEncoder(embed_dim=embed_dim)
        self.transformer = SimpleSpatiotemporalTransformer(embed_dim=embed_dim)
        self.attention_pool = TemporalAttentionPooling(in_channels=embed_dim)
        self.decoder = GenerativeDecoder(in_channels=embed_dim)

    def forward(self, viirs_seq, lulc):
        features = self.spatial_encoder(viirs_seq, lulc)
        attended_features = self.transformer(features)
        pooled_features, attn_weights = self.attention_pool(attended_features)
        
        # Raw linear output
        raw_predictions = self.decoder(pooled_features)
        
        # Sigmoid activation to strictly bound outputs between 0.0 and 1.0
        predictions = torch.sigmoid(raw_predictions)
        
        return predictions, attn_weights

class MaskedMSELoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, predictions, targets):
        # We apply MSE directly on the normalized [0, 1] values
        loss = F.mse_loss(predictions, targets)
        return loss

print("Architecture and MaskedMSELoss loaded successfully. Ready for training.")

Initializing the Hybrid CNN-Transformer Architecture...
Architecture and MaskedMSELoss loaded successfully. Ready for training.
